# 04. Classify stance

Labels each post as believing in, denying, or neutral toward human-caused climate
change, then weights that by the topic recall scores from notebook 03 to get the
topic-level stance `s_ij` of Eq. (3).

## The classifier emits four labels, not three

`robookwus/climate-change-belief` returns **believer**, **denier**, **neutral**, and
**factual news on climate change**. The 2025 run mapped the last of these onto
neutral, and the Methods section describes only three categories. That mapping is
defensible — a news report states a fact without taking a side — but it should be
stated, since a referee reading the model card will see four.

On the published sample the four came out at believer 61.4%, neutral 28.0%, denier
6.1%, news 4.4%. The Methods currently say "7% deniers", which rounds 6.1% the wrong
way; the corrected figures go in with the rest of this revision.

## Read the full text

As in notebooks 02 and 03, the classifier sees `full_text`. The 2025 run scored the
140-character `text`, which for 78% of posts is a sentence and a half plus a link.
Section 2 reproduces the old labels from the old input before switching.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [ ]:
!pip install -q transformers torch

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import config as cfg
import bertscore_fast as bf          # for pick_device

device = bf.pick_device()
print("device :", device)
print("model  :", cfg.STANCE_MODEL)
print("label -> score mapping:")
for k, v in cfg.STANCE_SCORE.items():
    print(f"   {v:+d}  {k}")

## 1. A batched classifier

`transformers.pipeline` one post at a time is slow for 590,000 posts. This does the
same thing in batches, sorted by length so each batch pads to roughly its own longest
member rather than to the longest post in the corpus.

In [ ]:
class StanceClassifier:
    """
    Batched text classification with the label set the model was trained on.

        clf = StanceClassifier()
        labels, scores = clf.predict(list_of_texts)
    """

    def __init__(self, model_name=None, device=None, batch_size=64, max_length=512):
        self.device = device or bf.pick_device()
        name = model_name or cfg.STANCE_MODEL
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSequenceClassification.from_pretrained(name)
        self.model.eval().to(self.device)
        self.id2label = self.model.config.id2label
        self.batch_size = batch_size
        self.max_length = max_length

    def predict(self, texts, progress_every=None):
        texts = [str(t) for t in texts]
        # Sort by length so batches pad to their own longest member, then restore order
        order = np.argsort([len(t) for t in texts])
        labels = [None] * len(texts)
        probs = np.zeros(len(texts), dtype=np.float32)

        t0 = time.time()
        for s in range(0, len(order), self.batch_size):
            idx = order[s:s + self.batch_size]
            enc = self.tokenizer([texts[i] for i in idx], padding=True,
                                 truncation=True, max_length=self.max_length,
                                 return_tensors="pt").to(self.device)
            with torch.no_grad():
                logits = self.model(**enc).logits
            p = torch.softmax(logits, dim=-1)
            best = p.argmax(dim=-1).cpu().numpy()
            conf = p.max(dim=-1).values.cpu().numpy()
            for j, i in enumerate(idx):
                labels[i] = self.id2label[int(best[j])]
                probs[i] = conf[j]
            if progress_every and (s // self.batch_size) % progress_every == 0:
                done = min(s + self.batch_size, len(texts))
                el = time.time() - t0
                print(f"  {done:>7,}/{len(texts):,}  {done/max(el,1e-9):.0f}/s", flush=True)
        return labels, probs


clf = StanceClassifier(device=device)
print("labels the model emits:", list(clf.id2label.values()))

## 2. Reproduce the published labels

Run the classifier over posts the 2025 run already labelled, using the same input it
used — the short `text` — and compare. Agreement should be essentially total; anything
less means the model or its version has moved, and that has to be settled before the
new labels mean anything.

In [ ]:
published = pd.read_csv(os.path.join(cfg.REVISION_DIR, 'work', '0531_Twitter_Stance.csv'),
                       usecols=['text', 'stance_label', 'stance_score'],
                       low_memory=False).sample(500, random_state=cfg.SEED)

got, _ = clf.predict(published.text.fillna('').tolist())
agree = np.mean(np.array(got) == published.stance_label.values)
print(f"label agreement with the 2025 run: {agree:.1%}  (n={len(published)})")

# and that the score mapping still reproduces their stance_score
mapped = pd.Series(got).map(cfg.STANCE_SCORE).values
print(f"score agreement: {np.mean(mapped == published.stance_score.values):.1%}")

assert agree > 0.98, "classifier output has drifted from the published labels"
print("\nReproduced.")

## 3. Classify the corpus

Now on `full_text`, which is the point of the revision.

In [ ]:
posts = pd.read_parquet(os.path.join(cfg.OUTPUT_DIR, 'climate_posts_2022.parquet'),
                        columns=['tweet_id', 'text', 'full_text', 'was_truncated'])
print(f"posts: {len(posts):,}")

RUN_FULL = False       # set True once section 2 passes

if RUN_FULL:
    labels, conf = clf.predict(posts.full_text.fillna('').tolist(), progress_every=50)
    stance = pd.DataFrame({
        'tweet_id': posts.tweet_id,
        'stance_label': labels,
        'score': conf,
    })
    stance['stance_score'] = stance.stance_label.map(cfg.STANCE_SCORE)
    stance.to_parquet(os.path.join(cfg.OUTPUT_DIR, 'stance_2022.parquet'), index=False)
    print(stance.stance_label.value_counts(normalize=True).round(4).to_string())
else:
    print("RUN_FULL is False.")

In [ ]:
### How much did reading the full text change the labels?

CHECK_N = 5000
if RUN_FULL:
    sub = posts[posts.was_truncated == 1].head(CHECK_N)
    short_labels, _ = clf.predict(sub.text.fillna('').tolist())
    full_labels = stance.set_index('tweet_id').loc[sub.tweet_id, 'stance_label'].values
    changed = np.mean(np.array(short_labels) != full_labels)
    print(f"truncated posts whose label changes with the full text: {changed:.1%} "
          f"(n={len(sub):,})")
    print(pd.crosstab(pd.Series(short_labels, name='on short text'),
                      pd.Series(full_labels, name='on full text'),
                      normalize=True).round(3).to_string())

## 4. Topic-weighted stance

`s_ij = O_i * r_ij`: a post's stance counts toward a topic in proportion to how much
it mentions that topic. A denier post that never touches carbon markets contributes
nothing to the stance on carbon markets.

In [ ]:
recall = pd.read_parquet(os.path.join(cfg.OUTPUT_DIR, 'topic_recall_2022.parquet'))
stance = pd.read_parquet(os.path.join(cfg.OUTPUT_DIR, 'stance_2022.parquet'))

merged = recall.merge(stance[['tweet_id', 'stance_score']], on='tweet_id', how='inner')
print(f"recall {len(recall):,}  stance {len(stance):,}  merged {len(merged):,}")
assert len(merged) == len(recall), "some posts lost their stance or their recall"

cols = cfg.tw_columns()
weighted = merged[cols].to_numpy() * merged.stance_score.to_numpy()[:, None]
out = pd.DataFrame(weighted, columns=[f'{c}_stance' for c in cols])
out.insert(0, 'tweet_id', merged.tweet_id.values)
out.to_parquet(os.path.join(cfg.OUTPUT_DIR, 'topic_stance_2022.parquet'), index=False)
out.iloc[:3, :5]

## 5. Stance over the year

Referee 1 asks whether the time of year moves support or dissent, pointing at the
November 2022 spike in climate coverage (R1.3.b). Plotting volume and stance on the
same axis answers it directly: on the published sample, volume rose sharply around
COP27 while mean stance barely moved.

In [ ]:
posts_dt = pd.read_parquet(os.path.join(cfg.OUTPUT_DIR, 'climate_posts_2022.parquet'),
                           columns=['tweet_id', 'created_at'])
m = posts_dt.merge(stance, on='tweet_id')
m['month'] = pd.to_datetime(m.created_at, utc=True, errors='coerce').dt.to_period('M')

monthly = m.groupby('month').agg(posts=('tweet_id', 'size'),
                                 mean_stance=('stance_score', 'mean'),
                                 believer=('stance_label', lambda s: (s.str.startswith('believer')).mean()),
                                 denier=('stance_label', lambda s: (s.str.startswith('denier')).mean()))
monthly.to_csv(os.path.join(cfg.OUTPUT_DIR, 'stance_by_month.csv'))
print(monthly.round(4).to_string())

fig_ax = monthly[['posts']].plot(kind='bar', figsize=(10, 3.5), legend=False,
                                 color='#C9D6DF', width=0.8)
ax2 = fig_ax.twinx()
monthly['mean_stance'].plot(ax=ax2, color='#2A6F97', marker='o')
fig_ax.set_ylabel('posts')
ax2.set_ylabel('mean stance')
fig_ax.set_title('Volume and stance through 2022')
fig_ax.set_xlabel('')